In [1]:
# ==============================================================================
# 2026改修版: SVM_Radial (設計基準 A〜K 完全準拠)
# - 重要度: Permutation Importance (R2 drop) を適用
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
})

set.seed(42)

# -------------------------------
# (F)(G) 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
file_names <- c(
  "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
  "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
  "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
  "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
)

# 出力先設定 (F)
today <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name <- "SVM_Radial"
run_dir <- file.path(out_root, paste0("Corr_1000_", model_name))
if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

# (J) Permutation Importance 関数
calc_svm_radial_importance <- function(model, data_scaled, target, features) {
  # ベースライン（全特徴量使用時）の予測値
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2 <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled
    temp[[col]] <- sample(temp[[col]]) # 特定の列のみシャッフル
    shuffled_pred <- tryCatch(predict(model, temp[, features, drop = FALSE]), error = function(e) NA)
    
    if (all(is.na(shuffled_pred))) {
      out_imps[[col]] <- 0
    } else {
      new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
      out_imps[[col]] <- max(0, base_r2 - new_r2) # 精度低下分を重要度とする
    }
  }
  data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) next
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL

  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) next

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # (C)(D)(E) 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]

    # (H) 多重共線性対策 (Corr > 0.99999)
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) next

    # (I) スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # (K) CV10設定 (A)OOD不要
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")
    tune_grid <- expand.grid(sigma = 2^seq(-15, 3, length = 7), C = c(1, 10, 100))

    # 学習
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE], y = df_scaled[[target]],
        method = "svmRadial", trControl = train_ctrl, tuneGrid = tune_grid, metric = "RMSE"
      ),
      error = function(e) { cat("  [ERROR] SVM_Radial failed:", e$message, "\n"); return(NULL) }
    )
    if (is.null(model)) next

    # --- 保存処理 ---
    tag <- paste0(tools::file_path_sans_ext(file), "_", target)
    
    # 1. モデルRDS
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))

    # 2. (B) CV10_OOF予測CSV
    pred_oof <- model$pred %>% 
      filter(sigma == model$bestTune$sigma, C == model$bestTune$C) %>%
      mutate(SampleID = rownames(df_scaled)[rowIndex], Type = "CV10_OOF") %>%
      select(SampleID, Observed = obs, Predicted = pred, Type)
    write.csv(pred_oof, file.path(run_dir, paste0(tag, "_pred_OOF.csv")), row.names = FALSE)

    # 3. (J) 重要度 (Permutation Importance)
    imp_df <- calc_svm_radial_importance(model, df_scaled, target, features)
    imp_df$File <- file
    imp_df$Target <- target
    importance_list[[length(importance_list)+1]] <- imp_df

    # 4. サマリー集計
    summary_list[[length(summary_list)+1]] <- data.frame(
      File = file, Target = target, 
      CV10_R2 = safe_r2(pred_oof$Observed, pred_oof$Predicted),
      CV10_RMSE = rmse(pred_oof$Observed, pred_oof$Predicted), 
      n_samples = nrow(df_scaled),
      n_features = length(features), 
      Best_Sigma = model$bestTune$sigma,
      Best_C = model$bestTune$C
    )
    
    cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f\n", file, target, tail(summary_list, 1)[[1]]$CV10_R2))
  }
}

# CSV出力
if (length(summary_list) > 0) {
  write.csv(do.call(rbind, summary_list), file.path(run_dir, "all_summary_CV10.csv"), row.names = FALSE)
  write.csv(do.call(rbind, importance_list), file.path(run_dir, "all_importance_SVM_Radial.csv"), row.names = FALSE)
}

cat("\n[DONE] SVM_Radial Analysis completed.\n")

  Finished: n_base.csv - Jsc | CV10_R2: 0.654
  Finished: n_base.csv - Voc | CV10_R2: 0.828
  Finished: n_base.csv - FF | CV10_R2: 0.471
  Finished: n_base.csv - PCEmax | CV10_R2: 0.619


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."


  Finished: n_base_OH_rebuilt.csv - Jsc | CV10_R2: 0.771


...

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

  Finished: m_all_FP_rebuilt.csv - FF | CV10_R2: 0.528


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

  Finished: m_all_FP_rebuilt.csv - PCEmax | CV10_R2: 0.729

[DONE] SVM_Radial Analysis completed.


In [2]:
# ==============================================================================
# 2026改修版: SVM_Radial (設計基準 A〜K 完全準拠)
# - 重要度: Permutation Importance (R2 drop) を適用
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
})

set.seed(42)

# -------------------------------
# (F)(G) 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("PCEmax")
file_names <- c(
"m_all_OH_rebuilt.csv"
)

# 出力先設定 (F)
today <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name <- "SVM_Radial"
run_dir <- file.path(out_root, paste0("Corr_1000_", model_name))
if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

# (J) Permutation Importance 関数
calc_svm_radial_importance <- function(model, data_scaled, target, features) {
  # ベースライン（全特徴量使用時）の予測値
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2 <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled
    temp[[col]] <- sample(temp[[col]]) # 特定の列のみシャッフル
    shuffled_pred <- tryCatch(predict(model, temp[, features, drop = FALSE]), error = function(e) NA)
    
    if (all(is.na(shuffled_pred))) {
      out_imps[[col]] <- 0
    } else {
      new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
      out_imps[[col]] <- max(0, base_r2 - new_r2) # 精度低下分を重要度とする
    }
  }
  data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) next
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL

  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) next

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # (C)(D)(E) 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]

    # (H) 多重共線性対策 (Corr > 0.99999)
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) next

    # (I) スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # (K) CV10設定 (A)OOD不要
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")
    tune_grid <- expand.grid(sigma = 2^seq(-15, 3, length = 7), C = c(1, 10, 100))

    # 学習
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE], y = df_scaled[[target]],
        method = "svmRadial", trControl = train_ctrl, tuneGrid = tune_grid, metric = "RMSE"
      ),
      error = function(e) { cat("  [ERROR] SVM_Radial failed:", e$message, "\n"); return(NULL) }
    )
    if (is.null(model)) next

    # --- 保存処理 ---
    tag <- paste0(tools::file_path_sans_ext(file), "_", target)
    
    # 1. モデルRDS
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))

    # 2. (B) CV10_OOF予測CSV
    pred_oof <- model$pred %>% 
      filter(sigma == model$bestTune$sigma, C == model$bestTune$C) %>%
      mutate(SampleID = rownames(df_scaled)[rowIndex], Type = "CV10_OOF") %>%
      select(SampleID, Observed = obs, Predicted = pred, Type)
    write.csv(pred_oof, file.path(run_dir, paste0(tag, "_pred_OOF.csv")), row.names = FALSE)

    # 3. (J) 重要度 (Permutation Importance)
    imp_df <- calc_svm_radial_importance(model, df_scaled, target, features)
    imp_df$File <- file
    imp_df$Target <- target
    importance_list[[length(importance_list)+1]] <- imp_df

    # 4. サマリー集計
    summary_list[[length(summary_list)+1]] <- data.frame(
      File = file, Target = target, 
      CV10_R2 = safe_r2(pred_oof$Observed, pred_oof$Predicted),
      CV10_RMSE = rmse(pred_oof$Observed, pred_oof$Predicted), 
      n_samples = nrow(df_scaled),
      n_features = length(features), 
      Best_Sigma = model$bestTune$sigma,
      Best_C = model$bestTune$C
    )
    
    cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f\n", file, target, tail(summary_list, 1)[[1]]$CV10_R2))
  }
}

# CSV出力
if (length(summary_list) > 0) {
  write.csv(do.call(rbind, summary_list), file.path(run_dir, "all_summary_CV10.csv"), row.names = FALSE)
  write.csv(do.call(rbind, importance_list), file.path(run_dir, "all_importance_SVM_Radial.csv"), row.names = FALSE)
}

cat("\n[DONE] SVM_Radial Analysis completed.\n")

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

In [3]:
# ==============================================================================
# 2026改修版: SVM_Radial (コンソール出力専用モデル)
# - 重要度: Permutation Importance (R2 drop)
# - 出力: 全てコンソールに表示 (ファイル保存なし)
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
})

set.seed(42)

# -------------------------------
# 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("PCEmax")
file_names <- c("m_all_OH_rebuilt.csv")

# 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

calc_svm_radial_importance <- function(model, data_scaled, target, features) {
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2 <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled
    temp[[col]] <- sample(temp[[col]])
    shuffled_pred <- tryCatch(predict(model, temp[, features, drop = FALSE]), error = function(e) NA)
    
    if (all(is.na(shuffled_pred))) {
      out_imps[[col]] <- 0
    } else {
      new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
      out_imps[[col]] <- max(0, base_r2 - new_r2)
    }
  }
  df <- data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
  return(df %>% arrange(desc(Importance)))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) {
    cat(sprintf("\n[ERROR] File not found: %s\n", filepath))
    next
  }
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next

  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) {
      cat(sprintf("\n[SKIP] Target %s not in %s\n", target, file))
      next
    }

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]

    # 多重共線性対策 (Corr > 0.99999)
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }

    # スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # CV10設定
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")
    tune_grid <- expand.grid(sigma = 2^seq(-15, 3, length = 7), C = c(1, 10, 100))

    # 学習
    cat(sprintf("\n--- Training: %s (Target: %s) ---\n", file, target))
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE], y = df_scaled[[target]],
        method = "svmRadial", trControl = train_ctrl, tuneGrid = tune_grid, metric = "RMSE"
      ),
      error = function(e) { cat("  [ERROR] SVM_Radial failed:", e$message, "\n"); return(NULL) }
    )
    if (is.null(model)) next

    # --- コンソールへの結果出力 ---
    
    # 1. 精度の表示
    pred_oof <- model$pred %>% 
      filter(sigma == model$bestTune$sigma, C == model$bestTune$C)
    
    r2_val <- safe_r2(pred_oof$obs, pred_oof$pred)
    rmse_val <- rmse(pred_oof$obs, pred_oof$pred)

    cat("\n[MODEL PERFORMANCE]\n")
    cat(sprintf("CV10 R2    : %.4f\n", r2_val))
    cat(sprintf("CV10 RMSE  : %.4f\n", rmse_val))
    cat(sprintf("Best Sigma : %.2e\n", model$bestTune$sigma))
    cat(sprintf("Best C     : %.1f\n", model$bestTune$C))
    cat(sprintf("n_samples  : %d\n", nrow(df_scaled)))
    cat(sprintf("n_features : %d\n", length(features)))

    # 2. 重要度の表示 (Top 20)
    cat("\n[VARIABLE IMPORTANCE (Top 20)]\n")
    imp_df <- calc_svm_radial_importance(model, df_scaled, target, features)
    print(head(imp_df, 20), row.names = FALSE)
    
    cat("\n", rep("=", 40), "\n")
  }
}

cat("\n[DONE] All analysis displayed above.\n")


--- Training: m_all_OH_rebuilt.csv (Target: PCEmax) ---


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."



[MODEL PERFORMANCE]
CV10 R2    : 0.9736
CV10 RMSE  : 0.4123
Best Sigma : 1.95e-03
Best C     : 100.0
n_samples  : 1045
n_features : 430

[VARIABLE IMPORTANCE (Top 20)]
                        Feature  Importance
                            Jsc 0.352730888
                             FF 0.109964691
      SMILESsnamep1M_P.T32EHiI. 0.007560984
         SMILESsnamep1M_P4TPDTQ 0.006172674
                            Voc 0.005422658
            SMILESsnamenM_DPc.T 0.005043911
 Additivesname_1.8octanedithiol 0.005024826
           SMILESsnamep1M_DTSC4 0.004633801
            SMILESsnamenM_MOPFP 0.004111370
            SMILESsnamep1M_F0F2 0.004004330
       SMILESsnamep1M_C3DPPTTTe 0.003778049
            SMILESsnamep1M_P3HT 0.003709873
             SMILESsnamenM_NCMM 0.003697859
         SMILESsnamep1M_PFDTDPP 0.003489191
        SMILESsnamep1M_DTPDPP3T 0.003253445
           SMILESsnamep1M_DTAnt 0.003132013
           Lay5electronodes1_Mg 0.002925081
        SMILESsnamep1M_C10DPPBP 0.00272